##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [5]:
!pip install patool
import patoolib

# Extract the dataset using patool instead of the command line 'unrar'
import os
if not os.path.exists('UCF11_updated_mpg'):
    print("Extracting UCF11 Dataset...")
    patoolib.extract_archive("UCF11_updated_mpg.rar", outdir=".")
else:
    print("Dataset already extracted!")

In [6]:
import os
import cv2
import math
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import yt_dlp
import matplotlib.pyplot as plt

# Set seeds for reproducibility
seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

# Configuration parameters
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
SEQUENCE_LENGTH = 20 # Number of frames to feed into the model
DATASET_DIR = "UCF11_updated_mpg"
CLASSES_LIST = ["basketball", "biking", "diving"]

In [7]:
# Convert labels to one-hot encoded vectors
one_hot_encoded_labels = to_categorical(labels)

# Split the dataset
features_train, features_test, labels_train, labels_test = train_test_split(
    features, one_hot_encoded_labels, test_size=0.2, shuffle=True, random_state=seed_constant
)

print(f"Training samples: {len(features_train)}")
print(f"Testing samples: {len(features_test)}")

In [9]:
# Install libraries for extracting rar files and downloading youtube videos
!pip install patool yt-dlp -q

import os
import cv2
import random
import urllib.request
import numpy as np
import patoolib
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Set random seeds for consistent results
seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

# Configuration parameters
IMAGE_HEIGHT, IMAGE_WIDTH = 64, 64
SEQUENCE_LENGTH = 20
DATASET_DIR = "UCF11_updated_mpg"
CLASSES_LIST = ["basketball", "biking", "diving"]

# Download and Extract Dataset
rar_file = "UCF11_updated_mpg.rar"
if not os.path.exists(rar_file):
    print("Downloading UCF11 Dataset...")
    urllib.request.urlretrieve("https://www.crcv.ucf.edu/data/UCF11_updated_mpg.rar", rar_file)
if not os.path.exists(DATASET_DIR):
    print("Extracting UCF11 Dataset...")
    patoolib.extract_archive(rar_file, outdir=".")

# Data Extraction Functions
def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    if video_frames_count == 0: return frames_list
    
    skip_frames_window = max(int(video_frames_count / SEQUENCE_LENGTH), 1)
    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read() 
        if not success: break
        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame / 255.0
        frames_list.append(normalized_frame)
    video_reader.release()
    return frames_list

def create_dataset():
    features, labels = [], []
    for class_index, class_name in enumerate(CLASSES_LIST):
        print(f'Extracting frames for Class: {class_name}...')
        class_dir_path = os.path.join(DATASET_DIR, class_name)
        for group_folder in os.listdir(class_dir_path):
            group_folder_path = os.path.join(class_dir_path, group_folder)
            if os.path.isdir(group_folder_path):
                for file_name in os.listdir(group_folder_path):
                    if file_name.endswith('.mpg'):
                        frames = frames_extraction(os.path.join(group_folder_path, file_name))
                        if len(frames) == SEQUENCE_LENGTH:
                            features.append(frames)
                            labels.append(class_index)
    return np.array(features), np.array(labels)

# Run Extraction and Split Data
features, labels = create_dataset()
print(f"Total videos processed: {len(features)}")

one_hot_encoded_labels = to_categorical(labels)
features_train, features_test, labels_train, labels_test = train_test_split(
    features, one_hot_encoded_labels, test_size=0.2, shuffle=True, random_state=seed_constant
)
print(f"Training samples: {len(features_train)} | Testing samples: {len(features_test)}")

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

def create_lrcn_model():
    model = Sequential()
    model.add(TimeDistributed(Conv2D(16, (3, 3), padding='same', activation='relu'), input_shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)))
    model.add(TimeDistributed(MaxPooling2D((4, 4))))
    model.add(TimeDistributed(Dropout(0.25)))
    
    model.add(TimeDistributed(Conv2D(32, (3, 3), padding='same', activation='relu')))
    model.add(TimeDistributed(MaxPooling2D((4, 4))))
    model.add(TimeDistributed(Dropout(0.25)))
    
    model.add(TimeDistributed(Conv2D(64, (3, 3), padding='same', activation='relu')))
    model.add(TimeDistributed(MaxPooling2D((2, 2))))
    model.add(TimeDistributed(Dropout(0.25)))
    
    model.add(TimeDistributed(Flatten()))
    model.add(LSTM(32))
    model.add(Dense(len(CLASSES_LIST), activation='softmax'))
    
    model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])
    return model

model = create_lrcn_model()
model.summary()

# Train the model
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=15, mode='min', restore_best_weights=True)
print("Starting Model Training...")
history = model.fit(x=features_train, y=labels_train, epochs=10, batch_size=4, 
                    shuffle=True, validation_split=0.2, callbacks=[early_stopping_callback])

# Plot the training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.legend()
plt.show()

# Evaluate and Save Model
model_evaluation_loss, model_evaluation_accuracy = model.evaluate(features_test, labels_test)
print(f"Test Accuracy: {model_evaluation_accuracy * 100:.2f}%")

# Mandatory Save Requirement
student_name = "HaneenAlhabib"
save_path = f"{student_name}_ucf11_model.h5"
model.save(save_path)
print(f"Model successfully saved as: {save_path}")